# 02 Silver: Joyce's cleaning version
**Input:** workspace.default.bronze_facilities (raw, unchanged)  
**Output:** workspace.default.silver_facilities_joyce

Steps:
1. Turn the JSON-text list fields (specialties, procedure, equipment, capability, websites...) into real lists and remove duplicates
2. Fix basic fields: valid facility/operator types, numbers, coordinates inside India, 6-digit PIN code
3. Add district + official state from the India Post PIN directory
4. **Flag** suspicious records (bad coordinates, state mismatch, duplicates, mixed identities, too many specialties, implausible counts)
5. Save the Silver table

We flag suspicious records instead of deleting them, so the Gold scoring step can give them a lower trust score. Only completely empty records are dropped.

In [0]:
# Step 1: load the raw data and turn the JSON-text list fields into real Python lists (deduplicated)
import json, re
import numpy as np
import pandas as pd

raw = spark.table("workspace.default.bronze_facilities").toPandas()
df = raw.copy()
print("raw rows:", len(df))

NULLS = {"", "null", "none", "nan", "[]", "{}"}

def clean_str(v):
    """Return a stripped string, or None for empty / 'null' values."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    s = str(v).strip()
    return None if s.lower() in NULLS else s

def parse_list(v):
    """'["a","a","b"]' -> ['a', 'b']  (keeps order, drops blanks and case-insensitive duplicates)"""
    s = clean_str(v)
    if s is None:
        return []
    try:
        items = json.loads(s)
        if not isinstance(items, list):
            items = [items]
    except Exception:
        items = [s]
    out, seen = [], set()
    for it in items:
        t = clean_str(it)
        if t and t.lower() not in seen:
            seen.add(t.lower())
            out.append(t)
    return out

def raw_len(v):
    try:
        x = json.loads(v)
        return len(x) if isinstance(x, list) else 1
    except Exception:
        return 0

LIST_COLS = ["specialties", "procedure", "equipment", "capability", "websites",
             "phone_numbers", "source_urls", "affiliationTypeIds"]
before = df["specialties"].apply(raw_len).sum()
for c in LIST_COLS:
    df[c] = df[c].apply(parse_list)
after = df["specialties"].str.len().sum()
print(f"specialty entries: {before:,} before dedup -> {after:,} after dedup")

raw rows: 10088
specialty entries: 279,006 before dedup -> 118,223 after dedup


In [0]:
# Step 2: fix basic fields: text, categories, numbers, coordinates, PIN code
for c in ["name", "description", "address_line1", "address_city", "address_stateOrRegion",
          "address_zipOrPostcode", "facilityTypeId", "operatorTypeId",
          "officialPhone", "officialWebsite", "email"]:
    df[c] = df[c].apply(clean_str)

# facility / operator type: keep only valid categories, fix typos (numbers and URLs become missing)
df["facility_type"] = df["facilityTypeId"].str.lower().replace({"farmacy": "pharmacy"})
df.loc[~df["facility_type"].isin(["hospital", "clinic", "dentist", "doctor", "pharmacy"]), "facility_type"] = None
df["operator_type"] = df["operatorTypeId"].str.lower().replace({"government": "public"})
df.loc[~df["operator_type"].isin(["private", "public"]), "operator_type"] = None

# numbers stored as text -> numeric; impossible values -> missing
df["n_doctors"] = pd.to_numeric(df["numberDoctors"], errors="coerce")
df["n_beds"] = pd.to_numeric(df["capacity"], errors="coerce")
df["year_established"] = pd.to_numeric(df["yearEstablished"], errors="coerce")
df.loc[~df["year_established"].between(1800, 2026), "year_established"] = np.nan
df.loc[~(df["n_doctors"] > 0), "n_doctors"] = np.nan
df.loc[~(df["n_beds"] > 0), "n_beds"] = np.nan

# coordinates must fall inside India's bounding box
df["lat"] = pd.to_numeric(df["latitude"], errors="coerce")
df["lon"] = pd.to_numeric(df["longitude"], errors="coerce")
df["flag_bad_coords"] = ~(df["lat"].between(6, 37.5) & df["lon"].between(68, 97.5))
df.loc[df["flag_bad_coords"], ["lat", "lon"]] = np.nan

# city / state: consistent casing; PIN code: exactly 6 digits
df["city"] = df["address_city"].str.strip().str.title()
df["state"] = df["address_stateOrRegion"].str.strip().str.title()
df["pincode"] = df["address_zipOrPostcode"].astype(str).str.extract(r"(\d{6})")[0]

print("valid facility_type:", df["facility_type"].notna().sum(), "| valid operator_type:", df["operator_type"].notna().sum())
print("bad / missing coordinates:", int(df["flag_bad_coords"].sum()), "| valid PIN codes:", df["pincode"].notna().sum())
print("doctors known:", df["n_doctors"].notna().sum(), "| beds known:", df["n_beds"].notna().sum(), "| year known:", df["year_established"].notna().sum())

valid facility_type: 9942 | valid operator_type: 9313
bad / missing coordinates: 124 | valid PIN codes: 9791
doctors known: 3617 | beds known: 2504 | year known: 4775


In [0]:
# Step 3: add district + official state from the India Post PIN directory
# The raw file has one row per POST OFFICE, so one PIN can appear many times.
# We aggregate to ONE row per PIN in Spark first, otherwise the join would duplicate facilities.
pin_agg = spark.sql("""
WITH p AS (
  SELECT regexp_extract(CAST(pincode AS STRING), '([0-9]{6})', 1) AS pincode,
         initcap(district) AS district, initcap(statename) AS state,
         try_cast(latitude AS DOUBLE) AS lat, try_cast(longitude AS DOUBLE) AS lon
  FROM workspace.default.bronze_india_post_pincode_directory),
counts AS (   -- most common district/state for each PIN
  SELECT pincode, district, state, COUNT(*) AS n,
         ROW_NUMBER() OVER (PARTITION BY pincode ORDER BY COUNT(*) DESC) AS rk
  FROM p WHERE pincode <> '' GROUP BY pincode, district, state),
stats AS (
  SELECT pincode, COUNT(DISTINCT district) AS pin_n_districts,
         percentile_approx(lat, 0.5) AS pin_lat, percentile_approx(lon, 0.5) AS pin_lon
  FROM p WHERE pincode <> '' GROUP BY pincode)
SELECT c.pincode, c.district AS pin_district, c.state AS pin_state,
       s.pin_n_districts, s.pin_lat, s.pin_lon
FROM counts c JOIN stats s USING (pincode)
WHERE c.rk = 1
""").toPandas()

df = df.drop(columns=[c for c in pin_agg.columns if c != "pincode" and c in df.columns])
df = df.merge(pin_agg, on="pincode", how="left")
assert len(df) == len(raw), "join changed the row count!"

# --- Fix the state field ---
# Many records put a CITY or DISTRICT in the state field (e.g. "Mumbai", "Pune", "Kollam"),
# or use an alias ("Nct Of Delhi"). Those are fixed, not flagged.
norm = lambda s: re.sub(r"[^a-z]", "", s.lower()) if isinstance(s, str) else None
VALID_STATES = {norm(s) for s in pin_agg["pin_state"].dropna().unique()}
ALIASES = {"nctofdelhi": "Delhi", "nctdelhi": "Delhi", "newdelhi": "Delhi", "delhincr": "Delhi",
           "orissa": "Odisha", "pondicherry": "Puducherry", "uttaranchal": "Uttarakhand",
           "jammukashmir": "Jammu And Kashmir"}

def fix_state(row):
    k = norm(row["state"])
    if k in ALIASES:
        return ALIASES[k]
    if k in VALID_STATES:
        return row["state"]
    # not a real state name -> use the PIN code's state (or missing if PIN unknown)
    return row["pin_state"] if isinstance(row["pin_state"], str) else None

df["state_raw"] = df["state"]
df["state"] = df.apply(fix_state, axis=1)
fixed = (df["state"].map(norm) != df["state_raw"].map(norm)) & df["state_raw"].notna()

# A REAL conflict: the record names a valid state, but its PIN code belongs to a different state
# (e.g. "Aravind Eye Hospital, Hyderabad, Telangana" with a Delhi PIN -> likely a merged record)
df["flag_state_mismatch"] = (df["pin_state"].notna() & df["state"].notna()
                             & (df["state"].map(norm) != df["pin_state"].map(norm)))

print("unique PINs in directory:", len(pin_agg))
print("facilities matched to a district:", df["pin_district"].notna().sum(), "of", len(df))
print("PINs that span more than one district:", int((df["pin_n_districts"] > 1).sum()))
print("state field fixed (held a city/district or an alias):", int(fixed.sum()))
print("real state conflicts (flagged):", int(df["flag_state_mismatch"].sum()))
display(df.loc[df["flag_state_mismatch"], ["name", "city", "state_raw", "state", "pincode", "pin_district", "pin_state"]].head(10))

unique PINs in directory: 19586
facilities matched to a district: 9572 of 10088
PINs that span more than one district: 513
state field fixed (held a city/district or an alias): 382
real state conflicts (flagged): 67


name,city,state_raw,state,pincode,pin_district,pin_state
Aravind Eye Hospital,Hyderabad,Telangana,Telangana,110002,Central,Delhi
Srinivasa Nursing Home,Adilabad,Andhra Pradesh,Andhra Pradesh,504001,Adilabad,Telangana
"Krishna Institute of Medical Sciences, Hyderabad",Hyderabad,Andhra Pradesh,Andhra Pradesh,500003,Hyderabad,Telangana
District Hospital Durg,Durg,Chhattisgarh,Chhattisgarh,262701,Kheri,Uttar Pradesh
Medicover Hospitals,Hyderabad,Telangana,Telangana,431003,Aurangabad,Maharashtra
Aditya Hospitals,Guntur,Andhra Pradesh,Andhra Pradesh,504215,Mancherial,Telangana
Care Hospitals,Hyderabad,Telangana,Telangana,530002,Visakhapatanam,Andhra Pradesh
Asha Hospital,Hyderabad,Andhra Pradesh,Andhra Pradesh,500029,Hyderabad,Telangana
Mayom Hospital,Gurgaon,Delhi,Delhi,122001,Gurugram,Haryana
"Primary Health Center, Katarkhatav",Katarkhatav,Maharashtra,Maharashtra,261405,Sitapur,Uttar Pradesh


In [0]:
# Step 4: flag suspicious records (we KEEP them, but Gold will trust them less)
text = (df["description"].fillna("") + " | " + df["capability"].apply(" | ".join)
        + " | " + df["procedure"].apply(" | ".join)).str.lower()
df["n_specialties"] = df["specialties"].str.len()
df["n_websites"] = df["websites"].str.len()   # kept as a number for Gold (long lists = noisier evidence), not a flag

# (a) empty record: no name, or no description / specialties / capability at all -> the only rows we drop
df["flag_empty"] = df["name"].isna() | (df["description"].isna() & (df["n_specialties"] == 0)
                                        & (df["capability"].str.len() == 0))

# (b) duplicate: same normalized name in the same city
df["_name_key"] = df["name"].fillna("").str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
df["flag_duplicate"] = (df["_name_key"] != "") & df.duplicated(subset=["_name_key", "city"], keep=False)

# (c) mixed identity: text describes a small single-system practice AND a large hospital
#     (sign that two different places were merged into one record, e.g. Dr Hedgewar Hospital)
small_practice = text.str.contains(r"homeopath|ayurved|unani|siddha|veterinar|dental clinic", regex=True)
large_hospital = (df["n_beds"] >= 100) | text.str.contains(r"open heart|cardiac surgery|neurosurgery|cath lab", regex=True)
df["flag_mixed_identity"] = small_practice & large_hospital

# (d) specialty inflation: 25+ distinct specialties but fewer than 50 beds known
df["flag_too_many_specialties"] = (df["n_specialties"] >= 25) & ~(df["n_beds"] >= 50)

# (e) implausible counts: huge numbers, or far more doctors than beds
df["flag_implausible_counts"] = ((df["n_doctors"] > 3000) | (df["n_beds"] > 5000)
                                 | ((df["n_beds"] >= 10) & (df["n_doctors"] > 5 * df["n_beds"])))

# (f) location check: facility coordinates vs. centre of its PIN code area (haversine distance, km)
def km(la1, lo1, la2, lo2):
    la1, lo1, la2, lo2 = (np.radians(x) for x in (la1, lo1, la2, lo2))
    a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

df["dist_to_pin_km"] = km(df["lat"], df["lon"], df["pin_lat"], df["pin_lon"]).round(1)
df["flag_coords_far_from_pin"] = df["dist_to_pin_km"] > 50

# When they disagree, a third source is the referee: the median location of facilities in the same city.
# (Diagnostic showed both sides are wrong sometimes: ~690 times the PIN, ~526 times the coordinates.)
city_centre = (df[df["lat"].notna()].groupby("city")[["lat", "lon"]].median()
               .rename(columns={"lat": "city_lat", "lon": "city_lon"}))
df = df.drop(columns=["city_lat", "city_lon"], errors="ignore").join(city_centre, on="city")
own_near_city = km(df["lat"], df["lon"], df["city_lat"], df["city_lon"]) <= 25
pin_near_city = km(df["pin_lat"], df["pin_lon"], df["city_lat"], df["city_lon"]) <= 25
has_own, has_pin, far = df["lat"].notna(), df["pin_lat"].notna(), df["flag_coords_far_from_pin"]

conditions = [
    has_own & has_pin & ~far,                    # both agree
    has_own & far & own_near_city & ~pin_near_city,   # PIN looks wrong
    has_pin & far & pin_near_city & ~own_near_city,   # coordinates look wrong
    has_own & ~has_pin,                          # nothing to check against
    ~has_own & has_pin,                          # only the PIN is available
    has_own & far,                               # conflict nobody can settle
]
sources = ["coordinates_match_pin", "coordinates_match_city", "pin_matches_city",
           "coordinates_unchecked", "pin_only", "uncertain"]
df["loc_source"] = np.select(conditions, sources, default="unknown")
use_pin = df["loc_source"].isin(["pin_matches_city", "pin_only"])
df["loc_lat"] = np.where(use_pin, df["pin_lat"], df["lat"])
df["loc_lon"] = np.where(use_pin, df["pin_lon"], df["lon"])
df["flag_location_uncertain"] = df["loc_source"].isin(["uncertain", "unknown"])

flag_cols = ["flag_bad_coords", "flag_state_mismatch", "flag_coords_far_from_pin", "flag_location_uncertain",
             "flag_duplicate", "flag_mixed_identity", "flag_too_many_specialties", "flag_implausible_counts"]
df["n_flags"] = df[flag_cols].sum(axis=1)

print("empty records (will be dropped):", int(df["flag_empty"].sum()))
for c in flag_cols:
    print(f"{c:28s} {int(df[c].sum()):5d}")
print("records with 0 flags:", int((df["n_flags"] == 0).sum()), "| 1 flag:", int((df["n_flags"] == 1).sum()),
      "| 2+ flags:", int((df["n_flags"] >= 2).sum()))
d = df["dist_to_pin_km"].dropna()
print(f"coords vs PIN centre: {len(d):,} comparable | median {d.median():.1f} km | within 10 km {(d <= 10).mean():.0%} | over 50 km {(d > 50).sum()}")
print("location source:", df["loc_source"].value_counts().to_dict())

empty records (will be dropped): 80
flag_bad_coords                124
flag_state_mismatch             67
flag_coords_far_from_pin      1582
flag_location_uncertain        455
flag_duplicate                  38
flag_mixed_identity            156
flag_too_many_specialties      436
flag_implausible_counts          7
records with 0 flags: 7844 | 1 flag: 1658 | 2+ flags: 586
coords vs PIN centre: 9,537 comparable | median 2.4 km | within 10 km 73% | over 50 km 1582
location source: {'coordinates_match_pin': 7955, 'coordinates_match_city': 690, 'pin_matches_city': 526, 'coordinates_unchecked': 427, 'uncertain': 366, 'unknown': 89, 'pin_only': 35}


In [0]:
# Step 5: keep the useful columns (snake_case names) and save the Silver table
rename = {"address_line1": "address", "officialPhone": "official_phone",
          "officialWebsite": "official_website"}
keep = ["unique_id", "name", "facility_type", "operator_type", "description",
        "specialties", "procedure", "equipment", "capability",
        "n_specialties", "n_doctors", "n_beds", "year_established",
        "address_line1", "city", "state", "state_raw", "pincode", "pin_district", "pin_state", "pin_n_districts",
        "lat", "lon", "pin_lat", "pin_lon", "dist_to_pin_km", "loc_lat", "loc_lon", "loc_source",
        "officialPhone", "officialWebsite", "email", "websites", "n_websites", "source_urls",
        ] + flag_cols + ["n_flags"]

silver = df.loc[~df["flag_empty"], keep].rename(columns=rename).reset_index(drop=True)
(spark.createDataFrame(silver)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable("workspace.default.silver_facilities_joyce"))

check = spark.table("workspace.default.silver_facilities_joyce")
print(f"saved workspace.default.silver_facilities_joyce: {check.count():,} rows x {len(check.columns)} columns")
print(f"rows: {len(raw):,} raw -> {len(silver):,} silver (dropped {int(df['flag_empty'].sum())} empty records)")
display(check.limit(5))

saved workspace.default.silver_facilities_joyce: 10,008 rows x 44 columns
rows: 10,088 raw -> 10,008 silver (dropped 80 empty records)


unique_id name facility_type operator_type description specialties procedure equipment capability n_specialties n_doctors n_beds year_established address city state state_raw pincode pin_district pin_state pin_n_districts lat lon pin_lat pin_lon dist_to_pin_km loc_lat loc_lon loc_source official_phone official_website email websites n_websites source_urls flag_bad_coords flag_state_mismatch flag_coords_far_from_pin flag_location_uncertain flag_duplicate flag_mixed_identity flag_too_many_specialties flag_implausible_counts n_flags b8a5401f-42f1-422a-8cd9-686a15b4cb76 Aravind Eye Hospital hospital private Referral hospital outside the state listed under Assam Government schemes (AAN/Atal Amrit Abhiyan). List(ophthalmology, internalMedicine, anesthesia, burnAndTraumaPlasticSurgery, otolaryngology, gynecologyAndObstetrics, neonatologyPerinatalMedicine, neurosurgery, orthopedicSurgery, pediatrics, pediatricSurgery, physicalMedicineAndRehabilitation, pulmonology, dermatology, generalSurgery, radiology, familyPlanningAndComplexContraception, infectiousDiseases, pathology, emergencyMedicine) List(Dialysis treatment is provided via a 20-bed Dialysis Unit with 10 machines as part of the Pradhan Mantri National Dialysis Programme, CT scan and MRI imaging services are available for registered IGH patients on advice, Routine minor and major operating theatre services are available, Provides Panchkarma therapies and OPD services at Charak Ayurvedic Kendra for joints (Osteoarthritis, Rheumatoid arthritis, Ankylosing spondylitis, PIVD cervical and lumbar), muscular disorders, neurological disorders (paralysis, migraine), and skin disorders (eczema, psoriasis)., Brain and spine surgical operations are performed in the Neurosurgery department, Subtraction angiography (DSA) is used for brain, spine and other regions, Interventional embolization procedures for tumours using DSA, Hemodialysis is performed in three units as a regular service, Peritoneal dialysis is available around the clock, A dedicated Dialysis Operating Theatre is commissioned in the OT Complex, Emergency department provides portable X-ray, Ultrasound, and ECG facilities in the Casualty room, Offers bronchoscopy with endobronchial ultrasound (EBUS)., Performs intercostal chest tube drainage and pleurodesis., Provides invasive and non-invasive mechanical ventilation., Performs overnight polysomnography and sleep-titration studies for sleep-disordered breathing., Offers cardio-pulmonary exercise testing (CPET)., Conducts lung function tests including lung volume studies, diffusion capacity measurements, and airway resistance assessments (Impulse Oscillometry)., Provides rapid on-site examination (ROSE) services., Uses portable bronchoscopy., Provides emergency nebulization., Upper GI Endoscopy, Colonoscopy, Endoscopic Biopsy, Bronchoscopy, Endobronchal Ultrasound, Thoracic Ultrasound, Pleural aspiration, CT Scan Chest, CT Coronary Angiography, CT Angiography, 2-D Echocardiography, Vascular Doppler Study, Ultrasound-guided interventional blocks, Fluoroscopy-guided interventional blocks, Platelet Rich Plasma (PRP) therapy, Radio-frequency ablation, Digital Substraction Angiography and Aneurysm Clipping, Complex surgeries – for skull base lesions, Aneuryms, AVMs, TransSpenoidal surgeries (TNTS) - for Craniopharyngioma, Pituitary adenoma, CSF leak, Craniotomy for neoplastic/other lesions at various locations, Spinal surgery for various pathologies, Peripheral nerve surgery, Ventriculo-peritoneal shunting, Endoscopy for intraventricular and deep seated lesions, Burr hole for biopsy, ICP monitoring, Ommaya reservoir placement, Emergency surgeries for head injury (SDH, EDH, Depressed fractures, ICH, cranioplasty, contusions, spinal trauma), Bedside procedures- EVD, Tracheostomy, Performs about 1,400 surgeries per year across major, emergency, and minor categories, OPD attendance more than 150,000 per year and IPD more than 30,000 per year, The Neurosurgical Department was started in 1999

In [0]:
# Diagnostic: when a facility's coordinates and its PIN code disagree (> 50 km), which one is wrong?
# Referee = city_lat / city_lon from Step 4: the median location of facilities in the same city.
far = df[df["flag_coords_far_from_pin"]].copy()
far["own_to_city_km"] = km(far["lat"], far["lon"], far["city_lat"], far["city_lon"]).round(1)
far["pin_to_city_km"] = km(far["pin_lat"], far["pin_lon"], far["city_lat"], far["city_lon"]).round(1)

own_ok = far["own_to_city_km"] <= 25
pin_ok = far["pin_to_city_km"] <= 25
print("records where coordinates and PIN disagree:", len(far))
print("  own coordinates agree with city, PIN does not ->", int((own_ok & ~pin_ok).sum()))
print("  PIN agrees with city, own coordinates do not ->", int((~own_ok & pin_ok).sum()))
print("  neither agrees with city                      ->", int((~own_ok & ~pin_ok).sum()))
print("\nlocation source used for distance search (whole table):")
print(df.loc[~df["flag_empty"], "loc_source"].value_counts().to_string())
display(far[["name", "city", "pincode", "pin_district", "dist_to_pin_km", "own_to_city_km", "pin_to_city_km", "loc_source"]]
        .sort_values("dist_to_pin_km", ascending=False).head(15))

records where coordinates and PIN disagree: 1582
  own coordinates agree with city, PIN does not -> 691
  PIN agrees with city, own coordinates do not -> 526
  neither agrees with city                      -> 365

location source used for distance search (whole table):
loc_source
coordinates_match_pin     7955
coordinates_match_city     690
pin_matches_city           526
coordinates_unchecked      427
uncertain                  366
pin_only                    35
unknown                      9


name,city,pincode,pin_district,dist_to_pin_km,own_to_city_km,pin_to_city_km,loc_source
Ibn Sina Hospital,Budgam,191111,Budgam,10623.4,0.0,10623.4,coordinates_match_city
Krishna Netralaya,Gurugram,122018,Gurugram,8748.8,5.4,8743.5,coordinates_match_city
Avya Skin Clinic,Gurugram,122018,Gurugram,8746.2,5.7,8743.5,coordinates_match_city
Cloudnine Golf Course Road - Gurgaon,Gurugram,122018,Gurugram,8745.3,1.8,8743.5,coordinates_match_city
Dentaire Smile Studio,Gurugram,122018,Gurugram,8744.5,2.0,8743.5,coordinates_match_city
"Esthetica International Clinique - 95P First Floor, Sector 47, Gurgaon, Haryana - 122018, India, South City ii, Gurugram",Gurugram,122018,Gurugram,8744.2,0.6,8743.5,coordinates_match_city
Ministry of smile,Gurugram,122018,Gurugram,8744.0,1.9,8743.5,coordinates_match_city
Polymedica Dental & Medical Centre,Gurgaon,122018,Gurugram,8743.9,4.1,8743.3,coordinates_match_city
Bloom Women & Kids Clinic,Gurgaon,122018,Gurugram,8743.8,4.4,8743.3,coordinates_match_city
Meraki Skin Clinic,Gurugram,122018,Gurugram,8743.8,0.9,8743.5,coordinates_match_city
